### 1.Load Bronze Data

In [0]:
silver_orders_df = spark.table("ecommerce_pipeline.bronze.bronze_orders")
silver_customers_df = spark.table("ecommerce_pipeline.bronze.bronze_customers")
silver_order_items_df = spark.table("ecommerce_pipeline.bronze.bronze_orders_items")
silver_payments_df = spark.table("ecommerce_pipeline.bronze.bronze_orders_payments")
silver_products_df = spark.table("ecommerce_pipeline.bronze.bronze_products")

### 2. Trim Spaces

In [0]:
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType

def trim_all_string_columns(df):
    for column in df.columns:
        if isinstance(df.schema[column].dataType, StringType):
            df = df.withColumn(column, trim(col(column)))
    return df

silver_customers_df = trim_all_string_columns(silver_customers_df)
silver_order_items_df = trim_all_string_columns(silver_order_items_df)
silver_orders_df = trim_all_string_columns(silver_orders_df)
silver_payments_df = trim_all_string_columns(silver_payments_df)
silver_products_df = trim_all_string_columns(silver_products_df)

### 3.Fix Date Formats

In [0]:
from pyspark.sql.functions import col, to_timestamp, coalesce

# Reusable parser for multiple date formats
def parse_timestamp(column):
    return coalesce(
        to_timestamp(col(column), "yyyy-MM-dd HH:mm:ss"),
        to_timestamp(col(column), "yyyy-MM-dd'T'HH:mm:ss"),
        to_timestamp(col(column), "dd-MM-yyyy HH:mm:ss"),
        to_timestamp(col(column), "yyyy/MM/dd HH:mm:ss"),
        to_timestamp(col(column), "dd/MM/yyyy HH:mm:ss")
    )

# -------------------------
# Fix silver_orders_df
# -------------------------
silver_orders_df = silver_orders_df \
    .withColumn("order_purchase_timestamp", parse_timestamp("order_purchase_timestamp")) \
    .withColumn("order_approved_at", parse_timestamp("order_approved_at")) \
    .withColumn("order_delivered_carrier_date", parse_timestamp("order_delivered_carrier_date")) \
    .withColumn("order_delivered_customer_date", parse_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", parse_timestamp("order_estimated_delivery_date")) \
    .withColumn("ingest_timestamp", parse_timestamp("ingest_timestamp"))

# -------------------------
# Fix silver_order_items_df
# -------------------------
silver_order_items_df = silver_order_items_df \
    .withColumn("shipping_limit_date", parse_timestamp("shipping_limit_date")) \
    .withColumn("ingest_timestamp", parse_timestamp("ingest_timestamp"))

# -------------------------
# Fix remaining tables (only ingest_timestamp)
# -------------------------
silver_customers_df = silver_customers_df.withColumn(
    "ingest_timestamp", parse_timestamp("ingest_timestamp")
)

silver_payments_df = silver_payments_df.withColumn(
    "ingest_timestamp", parse_timestamp("ingest_timestamp")
)

silver_products_df = silver_products_df.withColumn(
    "ingest_timestamp", parse_timestamp("ingest_timestamp")
)

# -------------------------
# Verification
# -------------------------
silver_orders_df.printSchema()
silver_order_items_df.printSchema()
silver_customers_df.printSchema()
silver_payments_df.printSchema()
silver_products_df.printSchema()

### 4.Complete Standardization

In [0]:
from pyspark.sql.functions import col, upper, trim, coalesce, lit, create_map
from pyspark.sql.types import StringType
from itertools import chain

# -------------------------
# Common function: trim + uppercase for all string columns
# -------------------------
def standardize_strings(df):
    return df.select([
        upper(trim(col(c))).alias(c) if isinstance(df.schema[c].dataType, StringType)
        else col(c)
        for c in df.columns
    ])

# -------------------------
# Apply to all tables
# -------------------------
silver_customers_df = standardize_strings(silver_customers_df)
silver_order_items_df = standardize_strings(silver_order_items_df)
silver_orders_df = standardize_strings(silver_orders_df)
silver_payments_df = standardize_strings(silver_payments_df)
silver_products_df = standardize_strings(silver_products_df)

# -------------------------
# Fix numeric columns
# -------------------------
silver_order_items_df = silver_order_items_df \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double"))

silver_payments_df = silver_payments_df \
    .withColumn("payment_value", col("payment_value").cast("double")) \
    .withColumn("payment_installments", col("payment_installments").cast("int"))

### 5.Standardize order_status values

In [0]:
from pyspark.sql.functions import when

silver_orders_df = silver_orders_df.withColumn(
    "order_status",
    when(col("order_status").isin("DELIVERED", "SHIPPED", "CANCELED"), col("order_status"))
    .otherwise("OTHER")
)

### 6.Remove Duplicates

In [0]:
silver_customers_df=silver_customers_df.dropDuplicates(["customer_id"])
silver_orders_df = silver_orders_df.dropDuplicates(["order_id"])
silver_order_items_df = silver_order_items_df.dropDuplicates(
    ["order_id", "order_item_id"]
)
silver_payments_df = silver_payments_df.dropDuplicates(
    ["order_id", "payment_sequential"]
)
silver_products_df = silver_products_df.dropDuplicates(["product_id"])

### 7.Handling Null values

In [0]:
from pyspark.sql.functions import col, when

# 1. CUSTOMERS
silver_customers_df = silver_customers_df \
    .withColumn("customer_city",
        when(col("customer_city").isNull(), "UNKNOWN").otherwise(col("customer_city"))
    ) \
    .withColumn("customer_state",
        when(col("customer_state").isNull(), "UNKNOWN").otherwise(col("customer_state"))
    )

# 2. ORDERS
silver_orders_df = silver_orders_df \
    .withColumn("order_status",
        when(col("order_status").isNull(), "UNKNOWN").otherwise(col("order_status"))
    )

# 3. ORDER ITEMS
silver_order_items_df = silver_order_items_df \
    .withColumn("price",
        when(col("price").isNull(), 0.0).otherwise(col("price"))
    ) \
    .withColumn("freight_value",
        when(col("freight_value").isNull(), 0.0).otherwise(col("freight_value"))
    )

# 4. PAYMENTS
silver_payments_df = silver_payments_df \
    .withColumn("payment_value",
        when(col("payment_value").isNull(), 0.0).otherwise(col("payment_value"))
    ) \
    .withColumn("payment_installments",
        when(col("payment_installments").isNull(), 0).otherwise(col("payment_installments"))
    ) \
    .withColumn("payment_type",
        when(col("payment_type").isNull(), "UNKNOWN").otherwise(col("payment_type"))
    )
# 5. PRODUCTS
silver_products_df = silver_products_df \
    .withColumn("product_category_name",
        when(col("product_category_name").isNull(), "UNKNOWN").otherwise(col("product_category_name"))
    )

In [0]:
silver_order_items_df.filter(col("price").isNull()).show()

In [0]:
_# 1. Check duplicates
from pyspark.sql.functions import countDistinct, count

silver_orders_df.select(count("*"), countDistinct("order_id")).show()

# 2. Check nulls
from pyspark.sql.functions import sum, col

silver_orders_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in silver_orders_df.columns
]).show()

# 3. Schema check
silver_orders_df.printSchema()

### 8.Use delta in datasets

In [0]:
silver_orders_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_pipeline.silver.silver_orders")

In [0]:
silver_customers_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ecommerce_pipeline.silver.silver_customers")

silver_order_items_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ecommerce_pipeline.silver.silver_order_items")

silver_payments_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ecommerce_pipeline.silver.silver_payments")

silver_products_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ecommerce_pipeline.silver.silver_products")